# Europarl EN to FR — Demo

Testing the outputs models from the LSTM and Transformer Experiements

In [1]:
!pip install -q sentencepiece

In [2]:
import os, math, sys, shutil, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm

print(f"Python : {sys.version.split()[0]}")
print(f"Torch  : {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}  "
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'})")

if not torch.cuda.is_available():
    print("\n!!! WARNING: no GPU detected. CPU inference of nn.Transformer can diverge from CUDA "
          "on recent torch versions. Switch the Colab runtime to GPU for correct outputs.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {device}")

PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
MAX_LEN = 64

Python : 3.12.13
Torch  : 2.10.0+cu128
CUDA   : True  (Tesla T4)

Device: cuda


In [4]:
DRIVE_BASE_DIR = '/content/drive/MyDrive/UB Reading Supp./Deep Learning/Group Project/model_assets'

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    print(f"Drive mounted. Reading run artifacts from:\n  {DRIVE_BASE_DIR}/m_training_<run>/")
except ImportError:
    print("Not running on Colab — skipping Drive mount.")
    print("Make sure (best.pt, spm.model) for each experiment is locally accessible.")

Mounted at /content/drive
Drive mounted. Reading run artifacts from:
  /content/drive/MyDrive/UB Reading Supp./Deep Learning/Group Project/model_assets/m_training_<run>/


In [5]:
# Test Transformer Models (Europarl_Training_Transformer.ipynb)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerNMT(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8,
                 num_enc=6, num_dec=6, dim_ff=2048, dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_enc, num_decoder_layers=num_dec,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.fc_out = nn.Linear(d_model, vocab_size, bias=False)
        self.fc_out.weight = self.embed.weight
    def _emb(self, x):
        return self.dropout(self.pos(self.embed(x) * math.sqrt(self.d_model)))
    @staticmethod
    def _causal(T, dev):
        return torch.triu(torch.ones(T, T, device=dev), diagonal=1).bool()
    def encode(self, src):
        src_pad = (src == PAD_ID)
        memory = self.transformer.encoder(self._emb(src), src_key_padding_mask=src_pad)
        return memory, src_pad
    def decode_step(self, tgt, memory, src_pad):
        tgt_pad = (tgt == PAD_ID)
        causal = self._causal(tgt.size(1), tgt.device)
        out = self.transformer.decoder(
            self._emb(tgt), memory,
            tgt_mask=causal,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=src_pad,
        )
        return self.fc_out(out)

print('Defined: TransformerNMT')

Defined: TransformerNMT


In [6]:
# Test LSTM (Europarl_Training_LSTM+attention.ipynb)
LSTM_EMBEDDING_DIM = 128
LSTM_HIDDEN_DIM    = 256
LSTM_NUM_LAYERS    = 2
LSTM_DROPOUT       = 0.3

class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = hidden.view(self.num_layers, 2, -1, self.hidden_dim)
        hidden = hidden.permute(1, 0, 2, 3).contiguous()
        hidden = hidden.view(2, -1, self.hidden_dim * 2)
        cell = cell.view(self.num_layers, 2, -1, self.hidden_dim)
        cell = cell.permute(1, 0, 2, 3).contiguous()
        cell = cell.view(2, -1, self.hidden_dim * 2)
        return outputs, hidden, cell

class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim * 2, num_layers,
                            batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden_dim * 4, 1)
        self.fc_out = nn.Linear(hidden_dim * 4, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.hidden_dim = hidden_dim
    def forward(self, x, hidden, cell, encoder_outputs):
        if x.dim() == 1:
            x = x.unsqueeze(1)
        embedded = self.dropout(self.embedding(x))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        seq_len = encoder_outputs.shape[1]
        hidden_repeated = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)
        concat = torch.cat((hidden_repeated, encoder_outputs), dim=2)
        attention_weights = torch.softmax(self.attention(concat), dim=1)
        context = torch.sum(attention_weights * encoder_outputs, dim=1).unsqueeze(1)
        concat_final = torch.cat((output, context), dim=2)
        prediction = self.fc_out(concat_final)
        return prediction.squeeze(1), hidden, cell

class Seq2SeqLSTM(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

print('Defined: LSTMEncoder, LSTMDecoder, Seq2SeqLSTM')

Defined: LSTMEncoder, LSTMDecoder, Seq2SeqLSTM


In [7]:
def _ids_to_text(ids, sp):
    out = []
    for i in ids:
        i = int(i)
        if i in (BOS_ID, PAD_ID): continue
        if i == EOS_ID: break
        out.append(i)
    return sp.decode(out)

def download_assets(drive_subfolder, suffix):
    drive_path = os.path.join(DRIVE_BASE_DIR, drive_subfolder)
    local_ckpt = f'best_{suffix}.pt'
    local_spm  = f'spm_{suffix}.model'
    print(f"Looking in: {drive_path}")
    for src_name, local_name in [('best.pt', local_ckpt), ('spm.model', local_spm)]:
        src = os.path.join(drive_path, src_name)
        if not os.path.exists(local_name):
            if not os.path.exists(src):
                raise FileNotFoundError(
                    f"{src} not found.\n"
                    f"  -> Run the corresponding training notebook to populate this folder, "
                    f"or upload {src_name} manually."
                )
            shutil.copy(src, local_name)
            print(f"  copied {src_name} -> {local_name}")
        else:
            print(f"  {local_name} already in working dir")
    return local_ckpt, local_spm

def load_transformer(suffix, drive_subfolder):
    local_ckpt, local_spm = download_assets(drive_subfolder, suffix)
    sp = spm.SentencePieceProcessor()
    sp.load(local_spm)
    vocab_size = sp.get_piece_size()
    model = TransformerNMT(vocab_size).to(device)
    ckpt = torch.load(local_ckpt, map_location=device, weights_only=False)
    state_key = 'Model' if 'Model' in ckpt else 'model'
    model.load_state_dict(ckpt[state_key], strict=True)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    em_std = model.embed.weight.std().item()
    trained_signal = em_std > 0.03
    print(f"  Transformer loaded: {n_params/1e6:.1f}M params")
    print(f"  epoch: {ckpt['epoch']}  val_loss: {ckpt['val_loss']:.4f}  embed.std: {em_std:.4f} "
          f"({'looks trained' if trained_signal else 'std low — possibly random init?'})")
    return {'model': model, 'sp': sp, 'arch': 'transformer', 'vocab_size': vocab_size,
            'epoch': ckpt['epoch'], 'val_loss': ckpt['val_loss']}

def load_lstm(suffix, drive_subfolder):
    local_ckpt, local_spm = download_assets(drive_subfolder, suffix)
    sp = spm.SentencePieceProcessor()
    sp.load(local_spm)
    vocab_size = sp.get_piece_size()
    enc = LSTMEncoder(vocab_size, LSTM_EMBEDDING_DIM, LSTM_HIDDEN_DIM, LSTM_NUM_LAYERS, LSTM_DROPOUT)
    dec = LSTMDecoder(vocab_size, LSTM_EMBEDDING_DIM, LSTM_HIDDEN_DIM, LSTM_NUM_LAYERS, LSTM_DROPOUT)
    model = Seq2SeqLSTM(enc, dec, device).to(device)
    ckpt = torch.load(local_ckpt, map_location=device, weights_only=False)
    state_key = 'Model' if 'Model' in ckpt else 'model'
    model.load_state_dict(ckpt[state_key], strict=True)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    em_std = model.encoder.embedding.weight.std().item()
    trained_signal = em_std > 0.02
    print(f"  LSTM loaded: {n_params/1e6:.1f}M params")
    print(f"  epoch: {ckpt['epoch']}  val_loss: {ckpt['val_loss']:.4f}  embed.std: {em_std:.4f} "
          f"({'looks trained' if trained_signal else 'std low — possibly random init?'})")
    return {'model': model, 'sp': sp, 'arch': 'lstm', 'vocab_size': vocab_size,
            'epoch': ckpt['epoch'], 'val_loss': ckpt['val_loss']}

@torch.no_grad()
def translate_transformer_greedy(model, sp, text, max_len=80):
    model.eval()
    ids = [BOS_ID] + sp.encode(text.lower(), out_type=int)[:MAX_LEN - 2] + [EOS_ID]
    src = torch.tensor(ids, device=device).unsqueeze(0)
    memory, src_pad = model.encode(src)
    ys = torch.tensor([[BOS_ID]], device=device)
    for _ in range(max_len - 1):
        logits = model.decode_step(ys, memory, src_pad)
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        ys = torch.cat([ys, nxt], dim=1)
        if nxt.item() == EOS_ID:
            break
    return _ids_to_text(ys[0].tolist(), sp)

@torch.no_grad()
def translate_transformer_beam(model, sp, text, vocab_size, beam=5, max_len=80, len_penalty=0.6):
    model.eval()
    ids = [BOS_ID] + sp.encode(text.lower(), out_type=int)[:MAX_LEN - 2] + [EOS_ID]
    src = torch.tensor(ids, device=device).unsqueeze(0)
    memory, src_pad = model.encode(src)
    memory_b = memory.expand(beam, -1, -1).contiguous()
    src_pad_b = src_pad.expand(beam, -1).contiguous()
    seqs = torch.full((beam, 1), BOS_ID, dtype=torch.long, device=device)
    scores = torch.full((beam,), -1e9, device=device); scores[0] = 0.0
    finished = []
    for _ in range(max_len - 1):
        logits = model.decode_step(seqs, memory_b, src_pad_b)
        logp = F.log_softmax(logits[:, -1], dim=-1)
        cand = scores.unsqueeze(1) + logp
        topk = cand.view(-1).topk(beam)
        beam_idx = topk.indices // vocab_size
        tok_idx  = topk.indices %  vocab_size
        seqs = torch.cat([seqs[beam_idx], tok_idx.unsqueeze(1)], dim=1)
        scores = topk.values
        is_eos = (tok_idx == EOS_ID)
        for b in is_eos.nonzero(as_tuple=True)[0].tolist():
            lp = ((5 + seqs.size(1)) / 6.0) ** len_penalty
            finished.append((scores[b].item() / lp, seqs[b].tolist()))
            scores[b] = -1e9
        if len(finished) >= beam:
            break
    if not finished:
        b = int(scores.argmax())
        finished.append((scores[b].item(), seqs[b].tolist()))
    finished.sort(key=lambda x: -x[0])
    return _ids_to_text(finished[0][1], sp)

@torch.no_grad()
def translate_lstm_greedy(model, sp, text, max_len=80):
    model.eval()
    src_ids = [BOS_ID] + sp.encode(text.lower(), out_type=int)[:MAX_LEN - 2] + [EOS_ID]
    src = torch.tensor([src_ids], device=device)
    encoder_outputs, hidden, cell = model.encoder(src)
    cur = torch.full((1,), BOS_ID, dtype=torch.long, device=device)
    out_ids = []
    for _ in range(max_len):
        output, hidden, cell = model.decoder(cur, hidden, cell, encoder_outputs)
        nxt = output.argmax(-1)
        tok = int(nxt[0])
        if tok == EOS_ID:
            break
        out_ids.append(tok)
        cur = nxt
    return sp.decode(out_ids)

@torch.no_grad()
def translate_lstm_beam(model, sp, text, vocab_size, beam=5, max_len=80, len_penalty=0.6):
    model.eval()
    src_ids = [BOS_ID] + sp.encode(text.lower(), out_type=int)[:MAX_LEN - 2] + [EOS_ID]
    src = torch.tensor([src_ids], device=device)
    encoder_outputs, hidden, cell = model.encoder(src)
    encoder_outputs_b = encoder_outputs.expand(beam, -1, -1).contiguous()
    hidden_b = hidden.expand(-1, beam, -1).contiguous()
    cell_b   = cell.expand(-1, beam, -1).contiguous()
    seqs = torch.full((beam, 1), BOS_ID, dtype=torch.long, device=device)
    scores = torch.full((beam,), -1e9, device=device); scores[0] = 0.0
    finished = []
    cur = seqs[:, -1]
    for _ in range(max_len - 1):
        output, hidden_b, cell_b = model.decoder(cur, hidden_b, cell_b, encoder_outputs_b)
        logp = F.log_softmax(output, dim=-1)
        cand = scores.unsqueeze(1) + logp
        topk = cand.view(-1).topk(beam)
        beam_idx = topk.indices // vocab_size
        tok_idx  = topk.indices %  vocab_size
        hidden_b = hidden_b[:, beam_idx, :].contiguous()
        cell_b   = cell_b[:, beam_idx, :].contiguous()
        seqs = torch.cat([seqs[beam_idx], tok_idx.unsqueeze(1)], dim=1)
        scores = topk.values
        is_eos = (tok_idx == EOS_ID)
        for b in is_eos.nonzero(as_tuple=True)[0].tolist():
            lp = ((5 + seqs.size(1)) / 6.0) ** len_penalty
            finished.append((scores[b].item() / lp, seqs[b].tolist()))
            scores[b] = -1e9
        if len(finished) >= beam:
            break
        cur = tok_idx
    if not finished:
        b = int(scores.argmax())
        finished.append((scores[b].item(), seqs[b].tolist()))
    finished.sort(key=lambda x: -x[0])
    return _ids_to_text(finished[0][1], sp)

def translate(experiment, text, mode='beam', beam=5):
    arch = experiment['arch']
    model, sp = experiment['model'], experiment['sp']
    if arch == 'transformer':
        if mode == 'beam':
            return translate_transformer_beam(model, sp, text, experiment['vocab_size'], beam=beam)
        return translate_transformer_greedy(model, sp, text)
    elif arch == 'lstm':
        if mode == 'beam':
            return translate_lstm_beam(model, sp, text, experiment['vocab_size'], beam=beam)
        return translate_lstm_greedy(model, sp, text)
    raise ValueError(f"unknown arch: {arch}")

SAMPLES = [
    "resumption of the session",
    "thank you for your attention",
    "i agree with the proposal",
    "climate change is a serious problem",
    "the european parliament meets today",
    "we must protect human rights and democratic values",
    "the commission has presented a new directive on energy efficiency",
    "i would like to thank the rapporteur for the excellent work",
]

# Single source of truth for which experiments to compare and the variable names that hold them.
EXPERIMENTS_REGISTRY = [
    ('Transformer 200k s42', 't200',     'transformer', 'm_training_200k',     't200k'),
    ('Transformer 200k s13', 't200_s13', 'transformer', 'm_training_200k_s13', 't200k_s13'),
    ('Transformer 200k s99', 't200_s99', 'transformer', 'm_training_200k_s99', 't200k_s99'),
    ('Transformer 500k s42', 't500',     'transformer', 'm_training_500k',     't500k'),
    ('LSTM + Attn 200k',     'lstm',     'lstm',        'm_training_lstm',     'lstm'),
]

def show_translations(experiment, label, samples=SAMPLES):
    print(f"=== {label} ===")
    print(f"checkpoint epoch={experiment['epoch']}  val_loss={experiment['val_loss']:.4f}  arch={experiment['arch']}\n")
    for s in samples:
        g = translate(experiment, s, mode='greedy')
        b = translate(experiment, s, mode='beam')
        print(f"  EN: {s}")
        print(f"  G : {g}")
        print(f"  B : {b}")
        print()

print('Helpers defined: download_assets, load_transformer, load_lstm, translate, show_translations')

Helpers defined: download_assets, load_transformer, load_lstm, translate, show_translations


## Transformer — 200k pairs, seed=42

In [8]:
t200 = load_transformer(suffix='t200k', drive_subfolder='m_training_200k')
show_translations(t200, 'Transformer 200k seed=42')

Looking in: /content/drive/MyDrive/UB Reading Supp./Deep Learning/Group Project/model_assets/m_training_200k
  copied best.pt -> best_t200k.pt
  copied spm.model -> spm_t200k.model


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


  Transformer loaded: 52.3M params
  epoch: 13  val_loss: 2.6490  embed.std: 0.0484 (looks trained)
=== Transformer 200k seed=42 ===
checkpoint epoch=13  val_loss=2.6490  arch=transformer

  EN: resumption of the session
  G : reprise de la session
  B : reprise de la session

  EN: thank you for your attention
  G : merci pour votre attention.
  B : merci pour votre attention.

  EN: i agree with the proposal
  G : je suis d'accord avec la proposition
  B : je suis d'accord avec la proposition

  EN: climate change is a serious problem
  G : le changement climatique est un problème grave.
  B : le changement climatique est un problème grave.

  EN: the european parliament meets today
  G : le parlement européen se réunit aujourd'hui
  B : le parlement européen se réunit aujourd'hui.

  EN: we must protect human rights and democratic values
  G : il faut protéger les droits de l'homme et les valeurs démocratiques
  B : il faut protéger les droits de l'homme et les valeurs démocratiques

## Transformer — 200k pairs, seed=13


In [9]:
t200_s13 = load_transformer(suffix='t200k_s13', drive_subfolder='m_training_200k_s13')
show_translations(t200_s13, 'Transformer 200k seed=13')

Looking in: /content/drive/MyDrive/UB Reading Supp./Deep Learning/Group Project/model_assets/m_training_200k_s13
  copied best.pt -> best_t200k_s13.pt
  copied spm.model -> spm_t200k_s13.model


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


  Transformer loaded: 52.3M params
  epoch: 11  val_loss: 2.6535  embed.std: 0.0469 (looks trained)
=== Transformer 200k seed=13 ===
checkpoint epoch=11  val_loss=2.6535  arch=transformer

  EN: resumption of the session
  G : reprise de la session
  B : reprise de la session

  EN: thank you for your attention
  G : merci de votre attention
  B : merci de votre attention

  EN: i agree with the proposal
  G : je suis d'accord avec la proposition
  B : je suis d'accord avec la proposition

  EN: climate change is a serious problem
  G : le changement climatique est un problème sérieux.
  B : le changement climatique est un problème sérieux.

  EN: the european parliament meets today
  G : le parlement européen se réunit aujourd'hui
  B : le parlement européen se réunit aujourd'hui

  EN: we must protect human rights and democratic values
  G : il faut protéger les droits de l'homme et les valeurs démocratiques
  B : nous devons protéger les droits de l'homme et les valeurs démocratique

## Transformer — 200k pairs, seed=99

In [10]:
t200_s99 = load_transformer(suffix='t200k_s99', drive_subfolder='m_training_200k_s99')
show_translations(t200_s99, 'Transformer 200k seed=99')

Looking in: /content/drive/MyDrive/UB Reading Supp./Deep Learning/Group Project/model_assets/m_training_200k_s99
  copied best.pt -> best_t200k_s99.pt
  copied spm.model -> spm_t200k_s99.model
  Transformer loaded: 52.3M params
  epoch: 14  val_loss: 2.6460  embed.std: 0.0488 (looks trained)
=== Transformer 200k seed=99 ===
checkpoint epoch=14  val_loss=2.6460  arch=transformer

  EN: resumption of the session
  G : reprise de la session
  B : reprise de la session

  EN: thank you for your attention
  G : merci de votre attention
  B : merci de votre attention

  EN: i agree with the proposal
  G : je suis d'accord avec la proposition
  B : je suis d'accord avec la proposition

  EN: climate change is a serious problem
  G : les changements climatiques sont un problème sérieux.
  B : le changement climatique est un problème sérieux.

  EN: the european parliament meets today
  G : le parlement européen se réunit aujourd'hui
  B : le parlement européen se réunit aujourd'hui

  EN: we m

## Transformer — 500k pairs, seed=42


In [11]:
t500 = load_transformer(suffix='t500k', drive_subfolder='m_training_500k')
show_translations(t500, 'Transformer 500k seed=42')

Looking in: /content/drive/MyDrive/UB Reading Supp./Deep Learning/Group Project/model_assets/m_training_500k
  copied best.pt -> best_t500k.pt
  copied spm.model -> spm_t500k.model
  Transformer loaded: 52.3M params
  epoch: 15  val_loss: 2.4182  embed.std: 0.0555 (looks trained)
=== Transformer 500k seed=42 ===
checkpoint epoch=15  val_loss=2.4182  arch=transformer

  EN: resumption of the session
  G : reprise de la session
  B : reprise de la session

  EN: thank you for your attention
  G : merci pour votre attention
  B : merci pour votre attention

  EN: i agree with the proposal
  G : je suis d'accord avec la proposition
  B : je suis d'accord avec la proposition

  EN: climate change is a serious problem
  G : le changement climatique est un problème grave.
  B : le changement climatique est un grave problème.

  EN: the european parliament meets today
  G : le parlement européen se réunit aujourd'hui
  B : le parlement européen se réunit aujourd'hui

  EN: we must protect huma

## LSTM + Attention — 200k pairs

In [12]:
lstm = load_lstm(suffix='lstm', drive_subfolder='m_training_lstm')
show_translations(lstm, 'LSTM + Attention 200k')

Looking in: /content/drive/MyDrive/UB Reading Supp./Deep Learning/Group Project/model_assets/m_training_lstm
  copied best.pt -> best_lstm.pt
  copied spm.model -> spm_lstm.model
  LSTM loaded: 26.3M params
  epoch: 32  val_loss: 5.2104  embed.std: 0.0695 (looks trained)
=== LSTM + Attention 200k ===
checkpoint epoch=32  val_loss=5.2104  arch=lstm

  EN: resumption of the session
  G : la session de session de la session de session de la session de la
  B : la reprise de la session de session de la session de session de

  EN: thank you for your attention
  G : je remercie remercie pour votre attention pour votre attention pour votre attention sur votre attention pour votre attention.
  B : je remercie remercie pour votre attention pour votre attention pour votre attention sur votre attention sur votre attention.

  EN: i agree with the proposal
  G : je suis d'accord avec la proposition de la proposition de la proposition de la proposition de la proposition de
  B : je suis d'accord a

## Comparison

In [13]:
loaded = []
for label, var_name, _arch, _drive, _suffix in EXPERIMENTS_REGISTRY:
    if var_name in globals():
        loaded.append((label, globals()[var_name]))

if not loaded:
    print('No experiments loaded yet.')
else:
    print(f"Comparing {len(loaded)} experiment(s) on the same sample sentences (beam=5):\n")
    print('-' * 80)
    for s in SAMPLES:
        print(f"EN : {s}")
        for label, exp in loaded:
            try:
                fr = translate(exp, s, mode='beam', beam=5)
                print(f"  [{label:<22}] {fr}")
            except Exception as e:
                print(f"  [{label:<22}] ERROR: {type(e).__name__}: {e}")
        print()

    print('\nCheckpoint summary:')
    for label, exp in loaded:
        print(f"  {label:<22}  epoch={exp['epoch']:>3}  val_loss={exp['val_loss']:.4f}  arch={exp['arch']}")

Comparing 5 experiment(s) on the same sample sentences (beam=5):

--------------------------------------------------------------------------------
EN : resumption of the session
  [Transformer 200k s42  ] reprise de la session
  [Transformer 200k s13  ] reprise de la session
  [Transformer 200k s99  ] reprise de la session
  [Transformer 500k s42  ] reprise de la session
  [LSTM + Attn 200k      ] la reprise de la session de session de la session de session de

EN : thank you for your attention
  [Transformer 200k s42  ] merci pour votre attention.
  [Transformer 200k s13  ] merci de votre attention
  [Transformer 200k s99  ] merci de votre attention
  [Transformer 500k s42  ] merci pour votre attention
  [LSTM + Attn 200k      ] je remercie remercie pour votre attention pour votre attention pour votre attention sur votre attention sur votre attention.

EN : i agree with the proposal
  [Transformer 200k s42  ] je suis d'accord avec la proposition
  [Transformer 200k s13  ] je suis d'ac

## Custom inputs


In [14]:
MY_INPUTS = [
    "the european union must take immediate action on climate policy",
    "the meeting will start at nine o'clock",
    "we need to find a balance between economic growth and environmental protection",
]

loaded = []
for label, var_name, _arch, _drive, _suffix in EXPERIMENTS_REGISTRY:
    if var_name in globals():
        loaded.append((label, globals()[var_name]))

if not loaded:
    print('No experiments loaded — run at least one section above first.')
else:
    for s in MY_INPUTS:
        print(f"EN: {s}")
        for label, exp in loaded:
            try:
                fr = translate(exp, s, mode='beam', beam=5)
                print(f"  [{label:<22}] {fr}")
            except Exception as e:
                print(f"  [{label:<22}] ERROR: {type(e).__name__}: {e}")
        print()

EN: the european union must take immediate action on climate policy
  [Transformer 200k s42  ] l'union européenne doit prendre des mesures immédiates en matière de politique du climat
  [Transformer 200k s13  ] l'union européenne doit prendre des mesures immédiates en matière de politique du climat
  [Transformer 200k s99  ] l'union européenne doit prendre des mesures immédiates en matière de politique du climat
  [Transformer 500k s42  ] l'union européenne doit agir immédiatement en matière de politique climatique
  [LSTM + Attn 200k      ] l'union européenne doit agir rapidement sur la plan politique de l'union européenne'union européenne

EN: the meeting will start at nine o'clock
  [Transformer 200k s42  ] la réunion débutera à neuf heures.
  [Transformer 200k s13  ] la réunion commencera à neuf heures.
  [Transformer 200k s99  ] la réunion débutera à neuf heures.
  [Transformer 500k s42  ] la réunion commencera à 9 heures
  [LSTM + Attn 200k      ] neuf neuf heures, la' la réunion